# 添加向量相似度显示的计分

# 一、先确认函数加在 retriever.py 里

路径：

F:\ResearchAgent\src\research_agent\rag\retriever.py

建议把它放在 retrieve_documents() 后面。

完整函数是：

def retrieve_documents_with_scores(
    query: str,
    task_type: str,
    top_k: int = 3,
    use_filter: bool = True,
):
    """
    检索文档，并返回相似度分数。

    返回格式：
    [
        (Document, score),
        (Document, score),
        ...
    ]

    注意：
    Chroma 返回的 score 通常是距离分数，不一定是“越大越相似”。
    一般情况下，分数越小表示距离越近、越相似。
    """
    vector_store = load_vector_store()

    metadata_filter = build_metadata_filter(task_type) if use_filter else None

    if metadata_filter:
        results = vector_store.similarity_search_with_score(
            query=query,
            k=top_k,
            filter=metadata_filter,
        )
    else:
        results = vector_store.similarity_search_with_score(
            query=query,
            k=top_k,
        )

    return results

注意这里的一个小坑：

Chroma 返回的 score 通常更接近“距离”，不是百分制相似度。
一般可以先理解为：score 越小，越相关。

# 二、新建测试脚本

创建：

F:\ResearchAgent\scripts\test_retriever_scores.py

写入：

from pathlib import Path
import sys


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))


from research_agent.rag.retriever import (
    build_metadata_filter,
    retrieve_documents_with_scores,
)


TEST_CASES = [
    {
        "query": "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？",
        "task_type": "dataset_recommendation",
    },
    {
        "query": "coco_val_n300_g1 这个实验的目的是什么？",
        "task_type": "experiment_analysis",
    },
    {
        "query": "Guardrail-Agnostic 这篇论文关注什么问题？",
        "task_type": "paper_question",
    },
]


def main():
    print("=" * 80)
    print("Test Retriever Scores")
    print("=" * 80)

    for case in TEST_CASES:
        query = case["query"]
        task_type = case["task_type"]

        metadata_filter = build_metadata_filter(task_type)

        print("\n" + "=" * 80)
        print(f"Query: {query}")
        print(f"Task Type: {task_type}")
        print(f"Metadata Filter: {metadata_filter}")
        print("=" * 80)

        results = retrieve_documents_with_scores(
            query=query,
            task_type=task_type,
            top_k=3,
            use_filter=True,
        )

        if not results:
            print("No results.")
            continue

        for i, (doc, score) in enumerate(results, start=1):
            print(f"\n[{i}] score = {score}")
            print("Source:", doc.metadata.get("path"))
            print("Source Type:", doc.metadata.get("source_type"))
            print("Title:", doc.metadata.get("title"))
            print("Dataset:", doc.metadata.get("dataset"))
            print("Run Tag:", doc.metadata.get("run_tag"))
            print("Chunk ID:", doc.metadata.get("chunk_id"))
            print("\nContent Preview:")
            print(doc.page_content[:300].replace("\n", " "))
            print("-" * 80)


if __name__ == "__main__":
    main()

# 三、运行测试

先确保你已经构建过索引：

cd F:\ResearchAgent
.\.conda\python.exe scripts\build_index.py

然后运行：

.\.conda\python.exe scripts\test_retriever_scores.py

# 四、你应该看到什么？

比如对于：

OpenImages-MIAP 的性别标注是图像级还是 bbox 级？

你应该看到类似：

================================================================================
Query: OpenImages-MIAP 的性别标注是图像级还是 bbox 级？
Task Type: dataset_recommendation
Metadata Filter: {'source_type': 'dataset_doc'}
================================================================================

[1] score = 0.42
Source: data/datasets/openimages_miap.md
Source Type: dataset_doc
Dataset: OpenImages-MIAP
Chunk ID: 18

Content Preview:
OpenImages-MIAP 是基于 OpenImages 的人物属性相关标注数据……

具体 score 数字不一定一样，这个正常。

你重点看三件事：

1. 有没有返回 3 条结果
2. Source Type 是否都是目标类型
3. score 排名靠前的结果内容是否更相关

# 五、如何判断测试成功？
成功标准 1：能正常运行，不报错

也就是没有：

ModuleNotFoundError
Chroma index not found
No module named ...
成功标准 2：metadata filter 生效

比如：

Task Type: dataset_recommendation
Metadata Filter: {'source_type': 'dataset_doc'}

下面结果应该都是：

Source Type: dataset_doc

如果 task_type 是：

experiment_analysis

结果应该都是：

Source Type: experiment_doc
成功标准 3：能看到 score

每条结果前面应该有：

[1] score = ...
[2] score = ...
[3] score = ...

这说明你确实调用的是：

similarity_search_with_score()

而不是普通的：

similarity_search()

# 六、如果你想对比“有过滤”和“无过滤”

你还可以把测试脚本改成这样，观察区别：

results = retrieve_documents_with_scores(
    query=query,
    task_type=task_type,
    top_k=3,
    use_filter=True,
)

改成：

results = retrieve_documents_with_scores(
    query=query,
    task_type=task_type,
    top_k=3,
    use_filter=False,
)

然后再运行一次。

区别是：

use_filter=True
只在对应 source_type 中检索

例如 dataset 问题只查：

dataset_doc
use_filter=False
查整个向量库

可能返回：

paper_note
experiment_doc
dataset_doc

这个对比非常适合你理解 Day 8 的意义。

# 七、推荐做一个对比版测试脚本

如果你想更清楚，可以新建：

scripts\compare_filter_scores.py

内容：

from pathlib import Path
import sys


PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))


from research_agent.rag.retriever import retrieve_documents_with_scores


def print_results(title, results):
    print("\n" + title)
    print("-" * 80)

    for i, (doc, score) in enumerate(results, start=1):
        print(f"[{i}] score={score}")
        print("source_type:", doc.metadata.get("source_type"))
        print("path:", doc.metadata.get("path"))
        print("preview:", doc.page_content[:150].replace("\n", " "))
        print()


def main():
    query = "OpenImages-MIAP 的性别标注是图像级还是 bbox 级？"
    task_type = "dataset_recommendation"

    print("=" * 80)
    print("Compare Retriever Scores With / Without Metadata Filter")
    print("=" * 80)
    print("Query:", query)
    print("Task Type:", task_type)

    filtered_results = retrieve_documents_with_scores(
        query=query,
        task_type=task_type,
        top_k=3,
        use_filter=True,
    )

    unfiltered_results = retrieve_documents_with_scores(
        query=query,
        task_type=task_type,
        top_k=3,
        use_filter=False,
    )

    print_results("With metadata filter", filtered_results)
    print_results("Without metadata filter", unfiltered_results)


if __name__ == "__main__":
    main()

运行：

.\.conda\python.exe scripts\compare_filter_scores.py

你可以看到：

With metadata filter
→ 应该都是 dataset_doc

Without metadata filter
→ 可能混入 paper_note / experiment_doc

这就是 Retriever Routing 的价值。

# 八、这个 score 以后有什么用？

现在只是调试用。

后面可以用于：

1. 判断检索结果是否足够相关
2. 如果 score 太差，提示“未找到可靠资料”
3. 给 evidence checker 提供参考
4. 在 debug 输出里展示检索质量
5. 比较不同 embedding 模型效果

但 Day 8 不要急着拿 score 做复杂决策。现在只要会打印、会观察就够了。

# 九、最短测试流程

你现在直接按这个来：

cd F:\ResearchAgent

.\.conda\python.exe scripts\build_index.py

.\.conda\python.exe scripts\test_retriever_scores.py

如果能看到：

score = ...
Source Type: dataset_doc / experiment_doc / paper_note

就说明 score 版本测试成功。